In [2]:
from utils import *

n_sim = 1000
n = int(1000.5/0.7)
B_RF = int(1000 )

seed = 42
n_covariates = 3

X_pred_point = pd.DataFrame({'X_1': [0], 'X_2': [1], 'X_3': [80], 'X_4': [40]})
data_generation_parameter =    { 'shape_weibull': 1,  'p_1': -0.405, 'p_2': -0.4, 'p_3': -0.05, 'p_4': -0.01, 'n': n,
                                    'scale_weibull_base':  15120        , 
                                    'rate_censoring':       0.023293   , 
                                    'tau': 37, 
                                    'X_pred_point': X_pred_point }



params_rf =         {   'n_estimators':B_RF,                        
                        'max_depth':4,
                        'min_samples_split':5,
                        'max_features': 'log2',
                        'random_state':  seed,
                        'weighted_bootstrapping': True, }

data = create_weibull_data(params=data_generation_parameter, random_state=seed)
df_train, df_test = stratified_split(data=data, random_state=seed, test_size=0.3)
df_train = create_data_with_ipc_weights(data=df_train, params=data_generation_parameter)
df_test = create_data_with_ipc_weights(data=df_test, params=data_generation_parameter)

### Random Forest Modell ###
# Fit
params_rf["random_state"] = seed
clf = DecisionTreeBaggingClassifier(params_rf)
clf.fit(
    X=df_train.drop(
        ["time", "event", "weights_ipcw", "survived"], axis=1
    ).values,
    y=df_train["survived"].values,
    sample_weights=df_train["weights_ipcw"].values,
)

# Evaluation auf Testdaten
_ , pred  =clf.predict_proba(df_test.drop(
    ["weights_ipcw", "survived", "time", "event"], axis=1
).values)
rf_test_mse = ipc_weighted_mse(
    y_true=df_test["survived"].values,
    y_pred=pred,
    sample_weight=df_test["weights_ipcw"].values,
)

# Prediction für X_erwartung
_ ,rf_pred = clf.predict_proba(data_generation_parameter['X_pred_point'].values)


In [16]:
X_pred_point = pd.DataFrame({'X_1': [1], 'X_2': [1], 'X_3': [89], 'X_4': [40]})
data_generation_parameter =    { 'shape_weibull': 1,  'p_1': -0.405, 'p_2': -0.4, 'p_3': -0.05, 'p_4': -0.01, 'n': n,
                                    'scale_weibull_base':  15120        , 
                                    'rate_censoring':       0.023293   , 
                                    'tau': 37, 
                                    'X_pred_point': X_pred_point }


calculate_true_survival_probability(data_generation_parameter)

0.4969956862016385

In [66]:
def calculate_ijk_jahn_variance(
    clf: DecisionTreeBaggingClassifier, X_pred_point: pd.DataFrame, df_train: pd.DataFrame
) -> float:

    T_N_b, pred = clf.predict_proba(X_pred_point.values)
    N_bi = clf.nbi
    weights = df_train["weights_ipcw"]
    B, n = N_bi.shape
    n_plus = np.sum(weights > 0)

    cov_i = ((N_bi - n * weights.values.reshape(1,-1)).T @ (T_N_b - pred)) / B
    cov_i_hoch2 = cov_i**2
    array = cov_i_hoch2/((weights.values.reshape(-1,1))**2)

    biased_var_estimate = np.sum(array[~np.isnan(array) & ~np.isinf(array)], axis=0) * (1/(np.sum(weights > 0))**2)

    #bias_correction1
    bias_correction =  (1/n_plus**2)  * np.var(T_N_b, axis=0, ddof=1)* n / B * np.sum( ( 1 / (weights[weights > 0] ) ) -1) 
    
    # bias correction 2
    bb = np.var((N_bi - n * weights.values.reshape(1,-1)) * (T_N_b - pred), axis=0, ddof=1)  / (weights**2).values.reshape(1,-1)
    bias_correction2 = (1/n_plus**2) * 1/B  * np.sum(bb[~np.isnan(bb) & ~np.isinf(bb)], axis=0)

    return biased_var_estimate , bias_correction[0], bias_correction2

calculate_ijk_jahn_variance(clf, X_pred_point, df_train)

(0.003958610196530734, 0.0033611176444649014, 0.00336315629978755)

In [36]:
T_N_b, pred = clf.predict_proba(X_pred_point.values)
N_bi = clf.nbi
weights = df_train["weights_ipcw"]
B, n = N_bi.shape

cov_i = ((N_bi - n * weights.values.reshape(1,-1)).T @ (T_N_b - pred)) / B
cov_i_hoch2 = cov_i**2
array = cov_i_hoch2/weights.values.reshape(-1,1)

biased_var_estimate = np.sum(array[~np.isnan(array) & ~np.isinf(array)], axis=0) * np.sum(weights**2)

bias_correction = n / B * np.sum(1-weights[weights > 0]) * np.var(T_N_b, axis=0, ddof=1)* np.sum(weights**2)
(N_bi - n * weights.values.reshape(1,-1)) * (T_N_b - pred)

array([[-0., -0.,  0.,  0.,  0.,  0.,  0.],
       [-0.,  0.,  0.,  0.,  0.,  0., -0.],
       [ 0.,  0.,  0.,  0.,  0.,  0., -0.]])

In [49]:
np.var((N_bi - n * weights.values.reshape(1,-1)) * (T_N_b - pred), axis=0, ddof=1) / weights.values.reshape(1,-1)

array([[ 0.,  0., nan,  0., nan, nan,  0.]])

In [48]:
weights.values.reshape(-1,1)

array([[0.28571429],
       [0.28571429],
       [0.        ],
       [0.28571429],
       [0.        ],
       [0.        ],
       [0.14285714]])

In [22]:
T_N_b - pred, np.array([[0.],
       [1.],
       [2.]])

(array([[0.],
        [0.],
        [0.]]),
 array([[0.],
        [1.],
        [2.]]))

array([[ 0.3125,  0.    , -1.125 , -0.6875, -1.125 ,  0.    ,  0.    ,
         2.3125,  0.3125],
       [-0.6875,  0.    , -0.125 ,  1.3125, -0.125 ,  0.    ,  0.    ,
        -0.6875,  0.3125],
       [-1.6875,  0.    , -0.125 ,  0.3125, -1.125 ,  0.    ,  0.    ,
         1.3125,  1.3125]])

In [19]:
n

9